# 04 — DeepEval + Graph Report

In [ ]:
import os
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.io as pio
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
)
from src.config import load_config

cfg = load_config()
judge_model = cfg["eval"]["Judge_model"]
judge_api_key = cfg["eval"].get("Judge_model_API_key")
if judge_api_key and judge_api_key != "<API_KEY>":
    os.environ.setdefault("OPENAI_API_KEY", judge_api_key)
responses = pd.read_parquet(cfg["paths"]["responses_parquet"])
print("Responses:", len(responses))

In [ ]:
eval_rows = []
for _, row in responses.iterrows():
    has_expected = pd.notna(row.get("expected_output"))
    tc = LLMTestCase(
        input=row["question"],
        actual_output=row["answer"],
        retrieval_context=row["retrieval_context"],
        expected_output=row["expected_output"] if has_expected else None,
    )

    metrics = [
        ("answer_relevancy", AnswerRelevancyMetric(model=judge_model)),
        ("faithfulness", FaithfulnessMetric(model=judge_model)),
        ("contextual_relevancy", ContextualRelevancyMetric(model=judge_model)),
    ]
    if has_expected:
        metrics += [
            ("contextual_precision", ContextualPrecisionMetric(model=judge_model)),
            ("contextual_recall", ContextualRecallMetric(model=judge_model)),
        ]

    for name, metric in metrics:
        metric.measure(tc)
        eval_rows.append({
            "question_id": row["question_id"],
            "config_id": row["config_id"],
            "dense_model": row["dense_model"],
            "sparse_model": row["sparse_model"],
            "late_model": row["late_model"],
            "llm_model": row["llm_model"],
            "metric": name,
            "score": metric.score,
            "reason": getattr(metric, "reason", None),
        })

eval_df = pd.DataFrame(eval_rows)
Path(cfg["paths"]["eval_details_parquet"]).parent.mkdir(parents=True, exist_ok=True)
eval_df.to_parquet(cfg["paths"]["eval_details_parquet"], index=False)
eval_df.to_json(cfg["paths"]["eval_details_jsonl"], orient="records", lines=True)

In [ ]:
summary = eval_df.groupby(
    ["config_id", "dense_model", "sparse_model", "late_model", "llm_model", "metric"]
)["score"].mean().reset_index()

heat = summary.pivot_table(index="config_id", columns="llm_model", values="score", aggfunc="mean")
fig = px.imshow(heat, text_auto=".3f", aspect="auto", title="18 Hybrid Retrieval Configurations x LLM")
Path(cfg["paths"]["eval_report_html"]).write_text(
    pio.to_html(fig, full_html=True, include_plotlyjs=True), encoding="utf-8"
)

leaderboard = summary.groupby("config_id")["score"].mean().sort_values(ascending=False)
print(leaderboard)